# CME SPAN-1 Margin Calculator — MCX Agri Commodity Portfolio

**Methodology:** CME Standard Portfolio Analysis of Risk (SPAN) — Level 1

---

### SPAN-1 Framework (Futures-Only Portfolio)

| Step | Component | Description |
|------|-----------|-------------|
| 1 | **Risk Arrays** | P&L across 16 price × volatility scenarios per contract |
| 2 | **Scanning Risk** | Worst-case loss (max loss across 16 scenarios) per Combined Commodity |
| 3 | **Inter-month Spread Charge** | Charge for same-commodity, different-expiry spreads |
| 4 | **Spot Month Charge** | Additional charge for positions in delivery/spot month |
| 5 | **Inter-commodity Spread Credit** | Credit for correlated commodity offsets |
| 6 | **SPAN Margin** | = Scanning Risk + Spread Charges − Inter-commodity Credits |
| 7 | **Net Option Value (NOV)** | N/A (futures-only portfolio) |
| 8 | **Performance Bond** | = SPAN Margin (since no short options) |

**Note:** Scanning Range inputs below are placeholders based on typical MCX initial margin percentages.  
**Always replace with the official MCX SPAN parameter file values before live use.**

## 0. Install & Import Libraries

In [ ]:
# Install openpyxl if not already present (needed for reading .xlsx)
!pip install openpyxl --quiet

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## 1. Load Portfolio from Excel

In [ ]:
# ── Upload the file in Colab before running this cell ──
# from google.colab import files
# uploaded = files.upload()   # uncomment to use Colab uploader

EXCEL_FILE = 'Portfolio_Agri.xlsx'   # <── adjust path if needed

raw = pd.read_excel(EXCEL_FILE)
raw.columns = raw.columns.str.replace('\n', ' ', regex=False).str.strip()

# Drop the portfolio total summary row
portfolio = raw[raw['Commodity'].notna() & (raw['Commodity'] != 'PORTFOLIO TOTAL')].copy()
portfolio = portfolio.rename(columns={
    'Lot Size (MT)':         'lot_size_MT',
    'Quote Unit':            'quote_unit',
    'Expiry Cycle':          'expiry_cycle',
    'Mar-26 Settle (INR)':  'settle_price',
    'Lots':                  'lots',
    'Direction':             'direction',
    'Notional (INR)':        'notional_INR'
})

# Signed lots: Long = +1, Short = -1
portfolio['signed_lots'] = portfolio.apply(
    lambda r: r['lots'] if r['direction'].strip().lower() == 'long' else -r['lots'], axis=1
)

# Contract value per lot = settle_price × lot_size_MT × conversion factor
# Quote unit mapping → units per MT
# 'per Quintal' → 10 quintals per MT → multiply by 10
# 'per 20 kg'  → 50 bags per MT     → multiply by 50
def units_per_lot(row):
    qu = str(row['quote_unit']).strip().lower()
    ls = row['lot_size_MT']
    if 'quintal' in qu:
        return ls * 10          # 1 MT = 10 quintals
    elif '20 kg' in qu:
        return ls * 50          # 1 MT = 50 × 20-kg bags
    else:
        return ls               # fallback: assume price is per MT

portfolio['units_per_lot'] = portfolio.apply(units_per_lot, axis=1)
portfolio['contract_value_INR'] = portfolio['settle_price'] * portfolio['units_per_lot']

display(portfolio[['Commodity','settle_price','units_per_lot','contract_value_INR',
                    'lots','direction','signed_lots','notional_INR']])
print(f'\nPortfolio loaded: {len(portfolio)} positions')

## 2. SPAN-1 Parameters

### 2A. Scanning Ranges per Commodity
Scanning Range (SR) = the maximum price move over the look-ahead period used by SPAN.
For MCX futures it is typically set ≈ 3× the daily 99% VaR (approximately 4–7% of contract value).

> **⚠️ Replace `scanning_range_pct` with the official MCX SPAN PA2 file values before production use.**

In [ ]:
# ─── EDITABLE INPUTS ────────────────────────────────────────────────────────
# Scanning Range as % of contract value (per lot)
# These proxy the official MCX SPAN scanning range; update from exchange PA file.
scanning_range_pct = {
    'Cotton Seed Oilcake':  5.00,   # %
    'Guar Seed':            5.00,
    'Guar Gum':             5.00,
    'Turmeric':             6.00,
    'Coriander (Dhaniya)':  6.00,
    'Castor Seed':          5.00,
    'Jeera (Cumin)':        6.00,
}

# Inter-commodity Spread Credits (pairs that get a margin offset)
# Format: (Commodity_A, Commodity_B, credit_pct_of_smaller_SR)
# Source: MCX SPAN inter-commodity spread table (illustrative values below)
inter_commodity_spreads = [
    ('Guar Seed', 'Guar Gum',            80),   # Strong crush spread credit
    ('Coriander (Dhaniya)', 'Turmeric',  20),   # Partial spice correlation
]

# Spot Month (Delivery Month) Charge — applied when position is in delivery month
# Set True if any position is in the delivery/spot month
spot_month_commodity = {}   # e.g., {'Turmeric': True} — set manually
spot_month_charge_pct = 25  # % surcharge on scanning risk for spot month positions
# ─────────────────────────────────────────────────────────────────────────────

# Compute Scanning Range per lot (INR)
portfolio['SR_pct'] = portfolio['Commodity'].map(scanning_range_pct)
portfolio['scanning_range_INR'] = portfolio['contract_value_INR'] * portfolio['SR_pct'] / 100

print('Scanning Range per lot (INR):')
display(portfolio[['Commodity','contract_value_INR','SR_pct','scanning_range_INR']])

## 3. Risk Arrays — 16 SPAN Scenarios

SPAN evaluates **16 scenarios** combining price moves (×−2, −1, −⅔, −⅓, 0, +⅓, +⅔, +1, +2) and volatility shifts:

| Scenario | Price Move | Vol Move | Weight |
|----------|-----------|----------|--------|
| 1–2 | ±1/3 SR | ±vol | 100% |
| 3–4 | ±2/3 SR | ±vol | 100% |
| 5–8 | ±1/3 SR | ±vol | 100% |
| 9–12 | ±full SR | ±vol | 100% |
| **13** | +2× SR (extreme up) | — | **35%** |
| **14** | −2× SR (extreme down) | — | **35%** |
| 15–16 | No price move | ±vol | 0% (futures only) |

For **futures-only** portfolios, vol scenarios produce zero P&L, so only price scenarios matter.

In [ ]:
# 16 SPAN scenarios: (price_move_fraction_of_SR, vol_shift_fraction, weight)
SCENARIOS = [
    #  ID   Price_frac  Vol_frac  Weight   Label
    ( 1,  +1/3,       +1,       1.00,  'Pr +1/3 SR, Vol Up'    ),
    ( 2,  +1/3,       -1,       1.00,  'Pr +1/3 SR, Vol Down'  ),
    ( 3,  -1/3,       +1,       1.00,  'Pr -1/3 SR, Vol Up'    ),
    ( 4,  -1/3,       -1,       1.00,  'Pr -1/3 SR, Vol Down'  ),
    ( 5,  +2/3,       +1,       1.00,  'Pr +2/3 SR, Vol Up'    ),
    ( 6,  +2/3,       -1,       1.00,  'Pr +2/3 SR, Vol Down'  ),
    ( 7,  -2/3,       +1,       1.00,  'Pr -2/3 SR, Vol Up'    ),
    ( 8,  -2/3,       -1,       1.00,  'Pr -2/3 SR, Vol Down'  ),
    ( 9,  +1,         +1,       1.00,  'Pr +Full SR, Vol Up'   ),
    (10,  +1,         -1,       1.00,  'Pr +Full SR, Vol Down' ),
    (11,  -1,         +1,       1.00,  'Pr -Full SR, Vol Up'   ),
    (12,  -1,         -1,       1.00,  'Pr -Full SR, Vol Down' ),
    (13,  +2,          0,       0.35,  'Pr +2× SR (Extreme Up)'),
    (14,  -2,          0,       0.35,  'Pr -2× SR (Extreme Dn)'),
    (15,   0,         +1,       1.00,  'No Price, Vol Up'      ),
    (16,   0,         -1,       1.00,  'No Price, Vol Down'    ),
]

scen_df = pd.DataFrame(SCENARIOS, columns=['Scenario','Price_Frac','Vol_Frac','Weight','Label'])

# ── Build Risk Array for each position ──────────────────────────────────────
# For a futures position:
#   P&L = signed_lots × price_move_INR_per_lot
#   price_move_INR_per_lot = Price_Frac × scanning_range_INR
# SPAN loss = −P&L  (positive = loss, negative = gain)
# Weighted loss for extreme scenarios (13, 14): weight = 0.35

results = []

for _, pos in portfolio.iterrows():
    commodity = pos['Commodity']
    signed_lots = pos['signed_lots']
    SR = pos['scanning_range_INR']   # INR per lot

    row = {'Commodity': commodity, 'signed_lots': signed_lots}
    losses = []
    for sid, pfrac, vfrac, weight, label in SCENARIOS:
        pnl_per_lot = pfrac * SR          # positive pfrac → price up → long gains
        total_pnl   = signed_lots * pnl_per_lot
        loss        = -total_pnl          # loss = negative of P&L
        wt_loss     = loss * weight       # apply scenario weight (only 35% for 13, 14)
        row[f'S{sid:02d}_{label[:12]}'] = round(wt_loss, 2)
        losses.append(wt_loss)

    row['Scanning_Risk_INR'] = max(losses)   # worst loss across 16 scenarios
    row['Active_Scenario']   = SCENARIOS[losses.index(max(losses))][4]
    results.append(row)

risk_df = pd.DataFrame(results)

# Display Risk Arrays (transposed for readability)
scenario_cols = [c for c in risk_df.columns if c.startswith('S')]
print('Risk Array (Loss in INR per position across 16 SPAN Scenarios):')
display(
    risk_df[['Commodity'] + scenario_cols + ['Scanning_Risk_INR','Active_Scenario']]
    .set_index('Commodity')
)

## 4. Scanning Risk Summary

In [ ]:
# Merge scanning risk back to portfolio
portfolio = portfolio.merge(
    risk_df[['Commodity','Scanning_Risk_INR','Active_Scenario']],
    on='Commodity', how='left'
)

total_scanning_risk = portfolio['Scanning_Risk_INR'].sum()

print('─' * 75)
print(f'{"Commodity":<25} {"Dir":>5} {"Lots":>5} {"SR%":>5}  {"Contract Value":>15}  {"Scanning Risk":>14}  Active Scenario')
print('─' * 75)
for _, r in portfolio.iterrows():
    print(f"{r['Commodity']:<25} {r['direction']:>5} {int(r['lots']):>5} "
          f"{r['SR_pct']:>4.1f}%  "
          f"{r['contract_value_INR']:>15,.0f}  "
          f"{r['Scanning_Risk_INR']:>14,.0f}  "
          f"{r['Active_Scenario']}")
print('─' * 75)
print(f'{"TOTAL SCANNING RISK":<55}  {total_scanning_risk:>14,.0f}')
print('─' * 75)

## 5. Inter-month Spread Charge
All positions here share the same expiry cycle (Monthly), so **no inter-month spread charge applies**  
(this section would activate if the same commodity had positions in different contract months).

In [ ]:
# Inter-month spread charge
# Each commodity has only one contract month → zero charge
intermonth_charge = 0.0

# If you had multi-month positions, you would compute:
#   inter_month_rate (INR per spread pair) × number_of_spread_pairs

print(f'Inter-month Spread Charge : INR {intermonth_charge:,.0f}')
print('  → All positions are in the same contract month (Mar-26). No charge.')

## 6. Spot Month / Delivery Month Charge
An additional charge is imposed on positions in or near the delivery month.  
Update `spot_month_commodity` in Section 2 if any position is in the spot month.

In [ ]:
spot_charge_total = 0.0
spot_details = []

for _, r in portfolio.iterrows():
    if spot_month_commodity.get(r['Commodity'], False):
        charge = r['Scanning_Risk_INR'] * spot_month_charge_pct / 100
        spot_charge_total += charge
        spot_details.append((r['Commodity'], charge))

if spot_details:
    for comm, chg in spot_details:
        print(f'  Spot Month Charge — {comm}: INR {chg:,.0f}')
else:
    print('  No positions flagged as spot/delivery month. Charge = INR 0')

print(f'Total Spot Month Charge   : INR {spot_charge_total:,.0f}')

## 7. Inter-commodity Spread Credits
SPAN grants margin offsets where two commodities are economically related  
(e.g., Guar Seed → Guar Gum crush spread; Coriander–Turmeric spice correlation).

In [ ]:
sr_lookup = dict(zip(portfolio['Commodity'], portfolio['Scanning_Risk_INR']))
dir_lookup = dict(zip(portfolio['Commodity'], portfolio['direction']))

total_ic_credit = 0.0
ic_rows = []

for comm_a, comm_b, credit_pct in inter_commodity_spreads:
    sr_a = sr_lookup.get(comm_a, 0)
    sr_b = sr_lookup.get(comm_b, 0)
    dir_a = dir_lookup.get(comm_a, '')
    dir_b = dir_lookup.get(comm_b, '')

    if sr_a == 0 or sr_b == 0:
        continue  # one leg not in portfolio

    # Credit applies only when directions are opposite (a natural spread/hedge)
    # or both same-direction (correlation credit still applies, but at reduced level)
    credit_base = min(sr_a, sr_b)          # credit on smaller leg's SR
    credit_INR  = credit_base * credit_pct / 100
    total_ic_credit += credit_INR

    ic_rows.append({
        'Leg A': comm_a, 'Dir A': dir_a, 'SR A (INR)': round(sr_a),
        'Leg B': comm_b, 'Dir B': dir_b, 'SR B (INR)': round(sr_b),
        'Credit %': f'{credit_pct}%',
        'Credit (INR)': round(credit_INR)
    })

if ic_rows:
    display(pd.DataFrame(ic_rows))
else:
    print('  No inter-commodity pairs found in portfolio.')

print(f'\nTotal Inter-commodity Spread Credit : INR {total_ic_credit:,.0f}')

## 8. Final SPAN-1 Margin Calculation

```
SPAN Margin = Scanning Risk
            + Inter-month Spread Charge
            + Spot Month Charge
            − Inter-commodity Spread Credit
```

Since this is a **futures-only** portfolio (no short options), Performance Bond = SPAN Margin.

In [ ]:
span_margin = (total_scanning_risk
               + intermonth_charge
               + spot_charge_total
               - total_ic_credit)

total_notional = portfolio['notional_INR'].sum()
margin_pct     = span_margin / total_notional * 100

print('=' * 60)
print('   CME SPAN-1 MARGIN SUMMARY — MCX AGRI PORTFOLIO')
print('=' * 60)
print(f'  Total Portfolio Notional (INR)  : {total_notional:>15,.0f}')
print(f'  Total Notional (INR Cr)         : {total_notional/1e7:>15.3f}')
print('─' * 60)
print(f'  (A) Scanning Risk               : {total_scanning_risk:>15,.0f}')
print(f'  (B) Inter-month Spread Charge   : {intermonth_charge:>15,.0f}')
print(f'  (C) Spot Month Charge           : {spot_charge_total:>15,.0f}')
print(f'  (D) Inter-commodity Credit      : {total_ic_credit:>15,.0f}')
print('─' * 60)
print(f'  SPAN MARGIN  = A+B+C−D (INR)   : {span_margin:>15,.0f}')
print(f'  SPAN MARGIN  (INR Lakh)         : {span_margin/1e5:>15.2f}')
print(f'  SPAN Margin as % of Notional    : {margin_pct:>15.2f}%')
print('=' * 60)

## 9. Position-level SPAN Margin Attribution

In [ ]:
summary = portfolio[['Commodity','direction','lots','settle_price',
                       'contract_value_INR','SR_pct',
                       'scanning_range_INR','Scanning_Risk_INR',
                       'Active_Scenario']].copy()

summary['Margin_as_%_Notional'] = (summary['Scanning_Risk_INR'] /
                                    (summary['contract_value_INR'] * summary['lots'])
                                    * 100).round(2)
summary['Weight_in_Total_SR_%'] = (summary['Scanning_Risk_INR'] /
                                    total_scanning_risk * 100).round(1)

summary = summary.rename(columns={
    'contract_value_INR':  'Contract_Val_per_Lot',
    'scanning_range_INR':  'SR_per_Lot_INR',
    'Scanning_Risk_INR':   'Total_Scanning_Risk_INR',
})

display(summary.set_index('Commodity'))

# ── Summary bar chart ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors = ['#2196F3' if d.strip().lower()=='long' else '#F44336'
          for d in portfolio['direction']]

# Chart 1: Scanning Risk by commodity
ax1 = axes[0]
bars = ax1.barh(portfolio['Commodity'], portfolio['Scanning_Risk_INR'],
                color=colors, edgecolor='white', height=0.6)
ax1.set_xlabel('Scanning Risk (INR)', fontsize=11)
ax1.set_title('Scanning Risk by Commodity\n(Blue = Long, Red = Short)', fontsize=12, fontweight='bold')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e5:.1f}L'))
for bar, val in zip(bars, portfolio['Scanning_Risk_INR']):
    ax1.text(val + 500, bar.get_y() + bar.get_height()/2,
             f'₹{val/1e5:.2f}L', va='center', fontsize=9)
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# Chart 2: % contribution to total scanning risk
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    portfolio['Scanning_Risk_INR'],
    labels=portfolio['Commodity'],
    autopct='%1.1f%%',
    colors=plt.cm.Set3.colors[:len(portfolio)],
    startangle=140,
    pctdistance=0.75
)
for t in autotexts:
    t.set_fontsize(9)
ax2.set_title('Scanning Risk Concentration\n(% of Total Scanning Risk)', fontsize=12, fontweight='bold')

plt.suptitle(f'MCX Agri Portfolio — SPAN-1 Margin  |  Total: ₹{span_margin/1e5:.2f} Lakh',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('SPAN1_Agri_Summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: SPAN1_Agri_Summary.png')

## 10. Sensitivity Analysis — SPAN Margin vs Scanning Range
How does total SPAN margin change if the exchange increases scanning ranges?

In [ ]:
SR_shocks = np.arange(0.80, 1.45, 0.05)   # 80% to 140% of base scanning range
margin_sens = []

for mult in SR_shocks:
    shocked_sr = portfolio['Scanning_Risk_INR'] * mult
    shocked_total_sr = shocked_sr.sum()
    shocked_ic = total_ic_credit * mult      # credits scale proportionally
    shocked_margin = shocked_total_sr - shocked_ic
    margin_sens.append({
        'SR_Multiplier': f'{mult:.0%}',
        'Total_SR (INR)': round(shocked_total_sr),
        'IC_Credit (INR)': round(shocked_ic),
        'SPAN_Margin (INR)': round(shocked_margin),
        'SPAN_Margin (Lakh)': round(shocked_margin / 1e5, 2),
        '% of Notional': round(shocked_margin / total_notional * 100, 2)
    })

sens_df = pd.DataFrame(margin_sens)
display(sens_df.set_index('SR_Multiplier'))

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sens_df['SR_Multiplier'], sens_df['SPAN_Margin (Lakh)'],
        marker='o', color='#1565C0', linewidth=2, markersize=7)
ax.axvline(x=sens_df[sens_df['SR_Multiplier']=='100%'].index[0],
           color='red', linestyle='--', alpha=0.6, label='Base Case (100%)')
ax.fill_between(sens_df.index, sens_df['SPAN_Margin (Lakh)'],
                alpha=0.15, color='#1565C0')
ax.set_xlabel('Scanning Range Multiplier', fontsize=11)
ax.set_ylabel('SPAN Margin (INR Lakh)', fontsize=11)
ax.set_title('SPAN Margin Sensitivity to Scanning Range Changes', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
for i, row in sens_df.iterrows():
    ax.annotate(f"{row['SPAN_Margin (Lakh)']:.1f}L",
                xy=(i, row['SPAN_Margin (Lakh)']),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('SPAN1_Sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: SPAN1_Sensitivity.png')

## 11. Export Full SPAN-1 Report to Excel

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

# ── Color palette ────────────────────────────────────────────────────────────
DARK_BLUE  = '1A237E'
MID_BLUE   = '1565C0'
LIGHT_BLUE = 'E3F2FD'
ORANGE     = 'E65100'
GREEN      = '2E7D32'
HEADER_FG  = 'FFFFFF'
GRAY_BG    = 'F5F5F5'

thin = Side(style='thin', color='BDBDBD')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def hdr(ws, row, col, val, bg=DARK_BLUE, fg=HEADER_FG, bold=True, size=10):
    c = ws.cell(row=row, column=col, value=val)
    c.font = Font(name='Arial', bold=bold, color=fg, size=size)
    c.fill = PatternFill('solid', start_color=bg)
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border = border
    return c

def val(ws, row, col, v, fmt=None, bold=False, bg=None, color='000000'):
    c = ws.cell(row=row, column=col, value=v)
    c.font = Font(name='Arial', size=10, bold=bold, color=color)
    c.border = border
    c.alignment = Alignment(horizontal='right' if isinstance(v, (int, float)) else 'left')
    if fmt: c.number_format = fmt
    if bg:  c.fill = PatternFill('solid', start_color=bg)
    return c

# ── Sheet 1: SPAN Summary ────────────────────────────────────────────────────
ws1 = wb.active
ws1.title = 'SPAN Summary'
ws1.column_dimensions['A'].width = 28
for col in 'BCDEFG': ws1.column_dimensions[col].width = 18
ws1.row_dimensions[1].height = 30

# Title
ws1.merge_cells('A1:G1')
c = ws1['A1']; c.value = 'CME SPAN-1 MARGIN REPORT — MCX AGRI COMMODITY PORTFOLIO'
c.font = Font(name='Arial', bold=True, size=13, color=HEADER_FG)
c.fill = PatternFill('solid', start_color=DARK_BLUE)
c.alignment = Alignment(horizontal='center', vertical='center')

ws1.merge_cells('A2:G2')
c2 = ws1['A2']; c2.value = 'Expiry: Mar-26  |  Settlement Date: Reference Date  |  Methodology: CME SPAN Level 1'
c2.font = Font(name='Arial', italic=True, size=10, color=MID_BLUE)
c2.alignment = Alignment(horizontal='center')

# Headers row 4
headers = ['Commodity','Direction','Lots','Settle Price (INR)',
           'Contract Value/Lot (INR)','Scanning Range %','Scanning Risk (INR)']
for i, h in enumerate(headers, 1):
    hdr(ws1, 4, i, h, bg=MID_BLUE)

# Data rows
for r_idx, row in enumerate(portfolio.itertuples(), 5):
    bg = GRAY_BG if r_idx % 2 == 0 else 'FFFFFF'
    val(ws1, r_idx, 1, row.Commodity, bg=bg)
    val(ws1, r_idx, 2, row.direction, bg=bg)
    val(ws1, r_idx, 3, int(row.lots), fmt='#,##0', bg=bg)
    val(ws1, r_idx, 4, row.settle_price, fmt='#,##0.00', bg=bg)
    val(ws1, r_idx, 5, row.contract_value_INR, fmt='#,##0', bg=bg)
    val(ws1, r_idx, 6, row.SR_pct/100, fmt='0.00%', bg=bg)
    val(ws1, r_idx, 7, row.Scanning_Risk_INR, fmt='#,##0', bg=bg, bold=True)

# Total row
tr = 5 + len(portfolio)
ws1.merge_cells(f'A{tr}:F{tr}')
c = ws1[f'A{tr}']; c.value = 'TOTAL SCANNING RISK'
c.font = Font(name='Arial', bold=True, size=10, color=HEADER_FG)
c.fill = PatternFill('solid', start_color=MID_BLUE)
c.alignment = Alignment(horizontal='right')
val(ws1, tr, 7, total_scanning_risk, fmt='#,##0', bold=True, bg=MID_BLUE, color=HEADER_FG)

# SPAN Margin summary block
sr = tr + 2
summary_data = [
    ('(A) Total Scanning Risk',             total_scanning_risk),
    ('(B) Inter-month Spread Charge',        intermonth_charge),
    ('(C) Spot Month Charge',                spot_charge_total),
    ('(D) Inter-commodity Spread Credit',   -total_ic_credit),
    ('SPAN MARGIN  (A+B+C+D)',               span_margin),
    ('SPAN Margin (INR Lakh)',               span_margin/1e5),
    ('SPAN Margin as % of Notional',         margin_pct/100),
]
fmts = ['#,##0','#,##0','#,##0','#,##0','#,##0','#,##0.00','0.00%']
for i, ((label, amount), fmt) in enumerate(zip(summary_data, fmts)):
    is_total = label.startswith('SPAN MARGIN')
    bg_val = ORANGE if is_total else (LIGHT_BLUE if i % 2 == 0 else 'FFFFFF')
    fg_col = HEADER_FG if is_total else '000000'
    ws1.merge_cells(f'A{sr+i}:F{sr+i}')
    c = ws1[f'A{sr+i}']
    c.value = label
    c.font = Font(name='Arial', bold=is_total, size=10, color=fg_col)
    c.fill = PatternFill('solid', start_color=bg_val)
    c.alignment = Alignment(horizontal='left')
    c.border = border
    cv = ws1.cell(row=sr+i, column=7, value=amount)
    cv.number_format = fmt
    cv.font = Font(name='Arial', bold=is_total, size=10, color=fg_col)
    cv.fill = PatternFill('solid', start_color=bg_val)
    cv.alignment = Alignment(horizontal='right')
    cv.border = border

# ── Sheet 2: Risk Arrays ─────────────────────────────────────────────────────
ws2 = wb.create_sheet('Risk Arrays')
ws2.column_dimensions['A'].width = 26
scen_labels = [s[4] for s in SCENARIOS]

ws2.merge_cells('A1:R1')
c = ws2['A1']; c.value = 'SPAN-1 Risk Arrays — P&L by Scenario (INR)  [Positive = Loss]'
c.font = Font(name='Arial', bold=True, size=12, color=HEADER_FG)
c.fill = PatternFill('solid', start_color=DARK_BLUE)
c.alignment = Alignment(horizontal='center', vertical='center')
ws2.row_dimensions[1].height = 25

hdr(ws2, 2, 1, 'Commodity', bg=MID_BLUE)
for i, label in enumerate(scen_labels, 2):
    hdr(ws2, 2, i, f'S{i-1:02d}\n{label[:14]}', bg=MID_BLUE)
    ws2.column_dimensions[get_column_letter(i)].width = 14
hdr(ws2, 2, len(scen_labels)+2, 'Scanning Risk', bg=ORANGE)
hdr(ws2, 2, len(scen_labels)+3, 'Active Scenario', bg=ORANGE)
ws2.column_dimensions[get_column_letter(len(scen_labels)+3)].width = 22

for r_idx, row in enumerate(risk_df.itertuples(), 3):
    bg = GRAY_BG if r_idx % 2 == 0 else 'FFFFFF'
    ws2.cell(row=r_idx, column=1, value=row.Commodity)
    sc = [c for c in risk_df.columns if c.startswith('S')]
    for ci, col in enumerate(sc, 2):
        v = getattr(row, col.replace(' ','_').replace('-','_').replace('/','_'), 0)
        c = ws2.cell(row=r_idx, column=ci, value=round(float(risk_df.loc[r_idx-3, col]),0))
        c.number_format = '#,##0'
        c.font = Font(name='Arial', size=9)
        c.border = border
        c.fill = PatternFill('solid', start_color=bg)
        if float(risk_df.loc[r_idx-3, col]) > 0:
            c.font = Font(name='Arial', size=9, color='C62828')
    sr_c = ws2.cell(row=r_idx, column=len(sc)+2, value=row.Scanning_Risk_INR)
    sr_c.number_format = '#,##0'
    sr_c.font = Font(name='Arial', bold=True, size=10)
    sr_c.fill = PatternFill('solid', start_color='FFF9C4')
    sr_c.border = border
    as_c = ws2.cell(row=r_idx, column=len(sc)+3, value=row.Active_Scenario)
    as_c.font = Font(name='Arial', size=9, color='1A237E')
    as_c.border = border

# ── Sheet 3: Scenario Reference ──────────────────────────────────────────────
ws3 = wb.create_sheet('Scenario Reference')
ws3.column_dimensions['A'].width = 10
ws3.column_dimensions['B'].width = 22
ws3.column_dimensions['C'].width = 16
ws3.column_dimensions['D'].width = 14
ws3.column_dimensions['E'].width = 10

hdr(ws3, 1, 1, 'Scenario', bg=DARK_BLUE)
hdr(ws3, 1, 2, 'Label', bg=DARK_BLUE)
hdr(ws3, 1, 3, 'Price Fraction of SR', bg=DARK_BLUE)
hdr(ws3, 1, 4, 'Vol Shift', bg=DARK_BLUE)
hdr(ws3, 1, 5, 'Weight', bg=DARK_BLUE)

for i, (sid, pfrac, vfrac, weight, label) in enumerate(SCENARIOS, 2):
    bg = 'FFF3E0' if weight < 1 else (GRAY_BG if i%2==0 else 'FFFFFF')
    ws3.cell(row=i, column=1, value=sid).border = border
    ws3.cell(row=i, column=2, value=label).border = border
    ws3.cell(row=i, column=3, value=pfrac).number_format='0.00'; ws3.cell(row=i,column=3).border=border
    ws3.cell(row=i, column=4, value=vfrac).border = border
    ws3.cell(row=i, column=5, value=weight).number_format='0%'; ws3.cell(row=i,column=5).border=border
    for col in range(1,6):
        ws3.cell(row=i, column=col).fill = PatternFill('solid', start_color=bg)

OUTPUT = 'SPAN1_Agri_Report.xlsx'
wb.save(OUTPUT)
print(f'SPAN-1 Report exported: {OUTPUT}')

---
## Notes & Assumptions

| Parameter | Value Used | Source / Note |
|-----------|-----------|---------------|
| Scanning Range | 5–6% of contract value | Proxy for MCX SPAN PA file; **update from MCX SPAN parameter file** |
| Guar Seed–Guar Gum IC Credit | 80% | Crush spread; confirm from MCX SPAN table |
| Coriander–Turmeric IC Credit | 20% | Spice correlation; confirm from MCX SPAN table |
| Inter-month spread charge | 0 | Single expiry month (Mar-26) only |
| Spot month charge | 0 | No positions flagged as delivery month |
| Extreme scenario weight | 35% | Standard CME SPAN specification |

> **Data Source:** Portfolio_Agri.xlsx (user-uploaded).  
> **SPAN PA File:** Obtain from MCX website → SPAN Parameters → Daily PA2 file for exact scanning ranges.